# 8. Reading rang-core's reports and judging an image

A `rang-core run` leaves a run directory full of JSON: the state of each
stage, one summary per stage, a receipt per command and GeMClean's image
report. This notebook reads each one, says which numbers to check first, and
then judges the image the run made with the checks an astronomer applies
before trusting it: noise against the thermal prediction, the residual's
statistics, dynamic range, the PSF, flux against known values and the stop
reason. The same checks are run on an image of uncalibrated data, so a
failing image is shown next to a passing one.

**The data** are A1 small, rang's simulated MeerKAT L-band observation
(16 antennas, 64 channels, 41 MB, written from seeds;
{gh}`research/a1-simtruth.md`), processed with E4's configuration
({gh}`examples/ms-to-image/config.yaml`, notebook {doc}`00-ms-to-image`).
The truth is known, so every check below is compared with it; on real data
the same checks are made against catalogues and the radiometer equation.
About a minute with 2 threads. Needs a source checkout (E4's configuration).

**Learning goals.** After this notebook you can

- find, in a run directory, what ran, with which configuration and code, and
  how long each stage took;
- read the flag, calibrate, apply and image summaries and spot a failed or
  suspicious stage;
- read GeMClean's `-report.json`: components, major cycles, stop reasons,
  restoring beam, residual statistics, memory;
- judge an image: thermal noise ratio, residual Gaussianity, dynamic range,
  PSF sidelobes, flux recovery, and whether CLEAN stopped for the right
  reason.

In [ ]:
%matplotlib inline
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("RAYON_NUM_THREADS", "2")  # the core's threads; set before the first core call

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS

from rang import core_ms
from rang.notebook import data

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "ms-to-image").is_dir())
WORK = data.default_cache_dir() / "reports"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
COMMANDS = {"rang-core": [sys.executable, "-m", "rang.core_cli"], "gemclean": [sys.executable, "-m", "rang.gemclean"], "python": [sys.executable]}


def sh(line, tail=1):
    # Run one command line in WORK as a shell would; print it and the last lines of its output.
    print("$", line)
    argv = shlex.split(line)
    p = subprocess.run(COMMANDS.get(argv[0], [argv[0]]) + argv[1:], cwd=WORK, capture_output=True, text=True, encoding="utf-8")
    lines = (p.stdout + p.stderr).rstrip().splitlines()
    print("\n".join(lines[-tail:] if tail else lines))
    assert p.returncode == 0, f"exit {p.returncode}"


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


sh("python -m rang.simtruth_ms generate small . --workers 2")
for name in ("config.yaml", "strategy.yaml"):
    shutil.copy(ROOT / "examples" / "ms-to-image" / name, WORK / name)
sh("rang-core run config.yaml a1-small.ms")
RUN = WORK / "a1-small.ms.rang-run"
truth = read(WORK / "truth.json")

## 1. What ran: `state.json`

The run directory `a1-small.ms.rang-run/` has one folder per stage and a
`state.json` that records, per stage, its status, its wall time, the hash
of the configuration sections it depends on and the flag version saved
before it wrote. **Check first:** every stage you asked for is `done`.
A stage that is `failed` or missing did not write its products, and a later
stage may have run on the earlier state of the MS.

In [ ]:
state = read(RUN / "state.json")
pd.DataFrame(
    [
        {"stage": s, "status": e["status"], "seconds": round(e["seconds"], 1), "config sha256": e["config_sha256"][:12], "flag version saved before": e.get("flag_version") or "–"}
        for s, e in state["stages"].items()
    ]
)

## 2. The stage summaries

Each stage writes `<stage>/summary.json`. Below are the numbers worth reading
in each, not every key; print the whole file when something looks wrong.

### 2.1 Flag

**Check:** the flagged fraction before and after. Compare it with what the
static mask alone removes and with earlier runs of the same configuration:
a fraction that jumps between runs, or one near 1, needs a look at the
strategy and the data before anything downstream is trusted.
The strategy and its static masks are recorded by hash, so two runs can be
compared exactly.

In [ ]:
flag = read(RUN / "flag" / "summary.json")
print(f"strategy {Path(flag['strategy']).name} (sha256 {flag['strategy_sha256'][:12]}), {flag['steps']} steps, write: {flag['write']}")
print(f"flagged before {flag['flagged_before']:.3f}, after {flag['flagged_after']:.3f} of all cells")

### 2.2 Calibrate

One entry per solved table. **Check:** every table was written, and the
fraction of solutions that converged and that are flagged. A converged
fraction well below 1, or many flagged solutions on one antenna, points at a
dead antenna, too little signal (`minsnr`) or too few baselines
(`minblperant`); a bad reference antenna shows as every antenna failing
together. The flux scale (`F`) gives the gain calibrator's fitted flux
density: compare it with a catalogue value when one exists.

In [ ]:
cal = read(RUN / "calibrate" / "summary.json")
rows = []
for s in cal["steps"]:
    row = {"table": s["name"], "type": s["type"], "rows": s.get("rows")}
    for k, v in (s.get("solutions") or [{}])[0].items():
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            row[k] = round(v, 4)
    rows.append(row)
display(pd.DataFrame(rows))
fs = next(s for s in cal["steps"] if s["type"] == "F")["fluxscale"]
for name, f in fs.items():
    print(f"fluxscale {name}: {f['fitted_flux_jy']:.3f} Jy at {f['reference_frequency_hz'] / 1e6:.0f} MHz")

Which antennas have flagged solutions, per table (CASA tables, read with
rang's own reader):

In [ ]:
names = [n for n in core_ms.open_table(str(WORK / "a1-small.ms" / "ANTENNA")).getcol("NAME")]
counts = {}
for s in cal["steps"]:
    path = RUN / "calibrate" / s["name"]
    if not (path / "table.dat").exists():
        continue
    with core_ms.open_table(str(path)) as t:
        if "FLAG" not in t.colnames():
            continue
        fl, ant = t.getcol("FLAG"), t.getcol("ANTENNA1")
    counts[s["name"]] = {names[a]: float(fl[ant == a].mean()) for a in np.unique(ant)}
flagged = pd.DataFrame(counts).fillna(0)
print(f"largest flagged fraction of any antenna in any table: {flagged.to_numpy().max():.3f}")
flagged.T.round(3)

### 2.3 Apply

**Check:** which tables went to which fields, the apply mode
(`calflag` flags data without a valid solution) and the warnings. An
interpolation over a gap longer than you expected, or a field that received
no tables, shows here.

In [ ]:
app = read(RUN / "apply" / "summary.json")
print(f"column {app['column']}, applymode {app['applymode']}, calwt {app['calwt']}, parang {app['parang']}")
print("warnings:", app["warnings"] or "none")
pd.DataFrame(
    [{k: (", ".join(map(str, v)) if isinstance(v, list) else v) for k, v in a.items() if not isinstance(v, dict)} for a in app["applied"]]
)

### 2.4 Image

The exact `gemclean` command line per field, its wall time and the SHA-256
of each file it wrote. Rerun the command line to reproduce the image.

In [ ]:
img = read(RUN / "image" / "summary.json")
for entry in img["images"]:
    print(f"{entry['field']}: {entry['wall_seconds']:.1f} s")
    print("  " + " ".join(entry["argv"][1:]))
    print("  files:", ", ".join(sorted(entry["files"])))

### 2.5 The receipts

`receipts/` holds one receipt per command (schema `rang.receipt`): the
arguments, the input hashes, the platform, rang's version, every `RANG_*`
variable and the git commit (null outside a checkout). **Check:** the
version and commit are the ones you think you ran, and no `RANG_*` variable
you forgot about is set: several change results
({doc}`../../cli`, "Environment variables").

In [ ]:
receipt = read(sorted((RUN / "receipts").glob("run-*.json"))[0])
env = receipt.get("environment", {})
print(f"schema {receipt['schema']} v{receipt['version']}, exit {receipt.get('exit')}, commit {str(receipt.get('commit'))[:12]}")
print("rang", env.get("rang_version"), "| Python", env.get("python"), "|", env.get("platform"))
set_vars = {k: v for k, v in (env.get("rang_environment") or {}).items() if v is not None}
print("RANG_* variables set:", set_vars or "none")

## 3. GeMClean's report

Next to the FITS images, `image-report.json` records the deconvolution:
the components, the major cycles and why each minor cycle stopped, the
stop reason of the whole run, the model flux, the residual's robust σ and
peak, the fitted restoring beam and the memory used.

**Check:** the run stopped on its threshold, not on `niter` (a run that
hits `niter` has not finished cleaning), and the residual peak is a small
multiple of the residual σ.

In [ ]:
folder = RUN / "image" / "SIM-TARGET"
rep = read(folder / "image-report.json")
r = rep["run"]
print("top-level keys:", ", ".join(rep))
print(f"{r['components']} components in {r['major_cycles']} major cycles; stop: {r['stop']}")
print(f"model flux {r['model_flux']:.3f} Jy; residual robust sigma {r['residual_mad_sigma'] * 1e3:.3f} mJy/beam, |residual| peak {r['residual_peak_abs'] * 1e3:.2f} mJy/beam ({r['residual_peak_abs'] / r['residual_mad_sigma']:.1f} sigma)")
print(f"restoring beam {rep['beam']['bmaj_deg'] * 3600:.1f}″ × {rep['beam']['bmin_deg'] * 3600:.1f}″")
pd.DataFrame([{"major cycle": c["index"], "components": c["components"], "first peak (mJy)": round(c["peak_start"] * 1e3, 2), "minor cycle ended by": c["stop"]} for c in r["cycles"]])

## 4. Judging the image

Seven checks, each one number. The functions below compute them from the
FITS files alone, so they apply to any image, not only rang's.

1. **Noise ratio:** the residual's robust σ (median absolute deviation ×
   1.4826, insensitive to sources) over the thermal noise the radiometer
   equation predicts for the imaged visibilities. Near 1 means the image is
   thermal-noise limited; well above 1 means calibration errors, unmodelled
   sources or RFI dominate.
2. **Gaussianity:** the fraction of residual pixels beyond 5σ against the
   Gaussian expectation (5.7e-7), and the excess kurtosis (0 for Gaussian
   noise). Non-Gaussian tails are artefacts.
3. **Dynamic range:** the restored peak over the residual σ.
4. **Local noise:** σ in a box around the brightest source over σ far
   from it. Calibration errors scale with source brightness, so they raise
   the noise around bright sources first.
5. **PSF:** the largest sidelobe outside the main lobe, and the restoring
   beam's axis ratio.
6. **Flux:** the restored peak at known point-source positions over the
   known flux density (here the truth; on real data a catalogue such as
   SUMSS or NVSS, or the calibrator's model).
7. **Stop:** whether CLEAN stopped on its threshold.

In [ ]:
def mad_sigma(x):
    x = np.asarray(x, float).ravel()
    return 1.4826 * np.median(np.abs(x - np.median(x)))


def load(path):
    with fits.open(path) as h:
        return np.squeeze(h[0].data).astype(float), h[0].header


def judge(prefix, report, points, thermal):
    restored, header = load(f"{prefix}-image.fits")
    residual, _ = load(f"{prefix}-residual.fits")
    psf, _ = load(f"{prefix}-psf.fits")
    wcs = WCS(header).celestial
    sigma = mad_sigma(residual)
    z = (residual - np.median(residual)) / sigma
    excess = float(np.mean(z**4) - 3)
    peak = float(restored.max())
    py, px = np.unravel_index(np.argmax(restored), restored.shape)
    box = residual[max(py - 25, 0) : py + 26, max(px - 25, 0) : px + 26]
    far = residual[: residual.shape[0] // 4, : residual.shape[1] // 4]
    # PSF: the main lobe is everything above half power connected to the centre;
    # its sidelobes are taken outside 3 beam widths (in pixels) of the centre.
    cy, cx = np.unravel_index(np.argmax(psf), psf.shape)
    yy, xx = np.indices(psf.shape)
    scale = abs(header["CDELT1"])
    width = 3 * header["BMAJ"] / scale
    sidelobe = float(np.abs(psf[np.hypot(yy - cy, xx - cx) > width]).max())
    ratios = []
    for p in points:
        x, y = (int(round(float(v))) for v in wcs.world_to_pixel_values(p["ra_deg"], p["dec_deg"]))
        if 0 <= x < restored.shape[1] and 0 <= y < restored.shape[0]:
            ratios.append(restored[y - 1 : y + 2, x - 1 : x + 2].max() / p["true_flux"])
    return {
        "noise ratio (robust σ / thermal)": round(sigma / thermal, 2),
        "robust σ (mJy/beam)": round(sigma * 1e3, 3),
        "pixels beyond 5σ (Gaussian 5.7e-7)": f"{np.mean(np.abs(z) > 5):.1e}",
        "excess kurtosis (Gaussian 0)": round(excess, 2),
        "dynamic range (peak / σ)": round(peak / sigma),
        "σ near the peak / σ far": round(mad_sigma(box) / mad_sigma(far), 2),
        "largest PSF sidelobe": round(sidelobe, 3),
        "beam axis ratio": round(header["BMAJ"] / header["BMIN"], 2),
        "point flux, restored / true (median)": round(float(np.median(ratios)), 3) if ratios else None,
        "CLEAN stop": report["run"]["stop"],
    }

The thermal noise of a naturally weighted Stokes I image is σ_vis / √(2 N),
with σ_vis the noise per real part of one correlation (A1 records it,
σ = SEFD / √(2 Δν Δt)), the factor 2 for XX and YY, and N the unflagged
cells. Briggs 0 weighting raises it by a factor set by the uv coverage
and the weights; the check below uses the natural-weighting value, so a
ratio somewhat above 1 is expected even for a perfect image (exercise 2
measures the factor).

The known point sources: A1's target components of kind `point`, with
their true flux at the image frequency (A1 has no primary beam, so these
are also the apparent fluxes).

In [ ]:
with core_ms.open_table(str(WORK / "a1-small.ms")) as t:
    field_id, flags = t.getcol("FIELD_ID"), t.getcol("FLAG")
target = [f["role"] for f in truth["fields"]].index("target")
good = ~flags[field_id == target][..., [0, 3]].any(-1)
thermal = truth["design"]["sigma_jy"] / np.sqrt(2 * good.sum())
nu = load(folder / "image-image.fits")[1]["CRVAL3"]
points = [
    {**c, "true_flux": c["flux"] * (nu / c["ref_freq_hz"]) ** c["alpha"]}
    for c in truth["target_components"]
    if c["kind"] == "point" and c["flux"] * (nu / c["ref_freq_hz"]) ** c["alpha"] > 20 * thermal
]
print(f"thermal noise (natural weighting): {thermal * 1e6:.1f} µJy/beam; {len(points)} point sources above 20× that")

Now the image of the calibrated data, and for contrast the same field
imaged by hand from the uncalibrated DATA column with the same settings:

In [ ]:
argv = img["images"][0]["argv"]
settings = " ".join(a for a in argv[1 : argv.index("-name")] + argv[argv.index("-name") + 2 : -1] if a != "-no-update-model-required")
sh(f"gemclean -name raw {settings} -data-column DATA -no-update-model-required a1-small.ms")
checks = pd.DataFrame(
    {
        "calibrated (the run)": judge(folder / "image", rep, points, thermal),
        "uncalibrated DATA": judge(WORK / "raw", read(WORK / "raw-report.json"), points, thermal),
    }
)
checks

**Reading the table.** The calibrated image passes the flux and PSF checks
and stops on its threshold, but its noise is several times the thermal
value and the noise rises around the brightest source. Notebook
{doc}`i7-imaging-and-deconvolution` shows why: A1 puts a gain error S on
the target alone, which the calibrators cannot measure and self-calibration
is meant to remove ({gh}`research/c5-selfcal.md`). The uncalibrated image
fails nearly every check. Dynamic range alone does not tell the two apart
as clearly as the noise ratio and the flux ratio do.

The residual histograms against a Gaussian of the robust σ:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, (label, prefix) in zip(axes, (("calibrated", folder / "image"), ("uncalibrated", WORK / "raw"))):
    res = load(f"{prefix}-residual.fits")[0]
    s = mad_sigma(res)
    bins = np.linspace(-8, 8, 161)
    ax.hist(res.ravel() / s, bins=bins, density=True, histtype="step", label="residual / robust σ")
    ax.plot(bins, np.exp(-(bins**2) / 2) / np.sqrt(2 * np.pi), "k-", lw=1, label="Gaussian")
    ax.set(yscale="log", ylim=(1e-7, 1), xlabel="σ", title=label)
    ax.legend(fontsize=8)
plt.show()

## 5. A short checklist

Before using an image from any pipeline:

| Check | Where | Healthy |
| --- | --- | --- |
| every stage `done` | `state.json` | no `failed`, no missing stage |
| flagged fraction | `flag/summary.json` | plausible for the band; stable between runs |
| solutions | `calibrate/summary.json`, the tables' FLAG | converged ≈ 1; no antenna mostly flagged |
| flux scale | `calibrate/summary.json` (F) | the gain calibrator near its catalogue flux |
| CLEAN stop | `image-report.json` | threshold, not `niter` |
| noise ratio | residual σ against the radiometer equation | near 1 (× the weighting's factor) |
| residual tails | pixels beyond 5σ, kurtosis | near Gaussian |
| local noise | σ near bright sources over σ far | near 1; above means calibration errors |
| flux | known sources against catalogue or model | within the calibration's accuracy |
| reproducibility | receipts, the `gemclean` argv, file hashes | the version, commit and `RANG_*` you expect |

## Exercises

1. Run `rang-core selfcal config.yaml a1-small.ms` after enabling the
   `selfcal` section and judge its image. Which checks improve, and which
   get worse? Compare with {gh}`research/c5b-selfcal-masks.md`.
2. Image with `-weight natural`. How much of the noise ratio was the
   weighting?
3. Flag one antenna entirely (FLAG on every row it is in) and rerun
   `calibrate`. Where does the summary show it, and what does `judge` say?
4. Change `-auto-threshold 3` to `-niter 200` without a threshold. What
   does the CLEAN stop check report, and how does the residual look?
5. Run `judge` on an image you made with WSClean of the same MS. Which
   numbers agree? See {gh}`research/g9-stokes.md` and
   {gh}`research/gemclean-speed.md` for how GeMClean was compared with WSClean.

## Where next

- {doc}`00-ms-to-image`: every output of the run, configuration section by
  section.
- {doc}`i7-imaging-and-deconvolution`: the noise against the truth-corrected
  data, and why the target's residual is not thermal.
- {doc}`09-cpu-and-gpu`: which parts run on which hardware, and how the
  paths are checked against each other.
- {doc}`../../cli`: every command, option and `RANG_*` variable.